In [3]:
# from vnstock import Vnstock
# import pandas as pd
# import time

# symbols = [
#     "HPG", "VCB", "SSI", "VNM", "MWG",
#     "FPT", "GAS", "VPB", "TCB", "MBB",
#     "ACB", "VIC", "VHM", "BID", "CTG",
#     "PNJ", "REE", "HDB", "VRE", "PLX",
#     "POW", "BVH", "GVR", "KDH", "NVL",
#     "PDR", "STB", "MSN", "SAB", "DGC"
# ]

# vn = Vnstock()
# all_data = []

# REQUEST_DELAY = 3.5  # 👈 quan trọng

# for symbol in symbols:
#     print(f"Đang lấy dữ liệu {symbol}...")

#     stock = vn.stock(symbol=symbol, source="VCI")

#     for func, name in [
#         (stock.finance.income_statement, "income_statement"),
#         (stock.finance.balance_sheet, "balance_sheet"),
#         (stock.finance.cash_flow, "cash_flow"),
#     ]:
#         df = func(period="year")
#         df["symbol"] = symbol
#         df["report"] = name
#         all_data.append(df)

#         time.sleep(REQUEST_DELAY)  # 👈 chặn rate limit

# final_df = pd.concat(all_data, ignore_index=True)
# final_df.to_excel("financial_report_30_companies.xlsx", index=False)

# print("Hoàn thành!")

In [4]:
from vnstock import Vnstock
import pandas as pd
import time

symbols = [
    "HPG", "VCB", "SSI", "VNM", "MWG",
    "FPT", "GAS", "VPB", "TCB", "MBB",
    "ACB", "VIC", "VHM", "BID", "CTG",
    "PNJ", "REE", "HDB", "VRE", "PLX",
    "POW", "BVH", "GVR", "KDH", "NVL",
    "PDR", "STB", "MSN", "SAB", "DGC"
]

vn = Vnstock()
all_data = []

# Chỉ cần delay ngắn nếu dùng code xử lý lỗi bên dưới
REQUEST_DELAY = 3.5

for symbol in symbols:
    print(f"🔄 Đang xử lý mã: {symbol}...")
    
    # Khởi tạo object stock
    stock = vn.stock(symbol=symbol, source="VCI")

    for func, name in [
        (stock.finance.income_statement, "income_statement"),
        (stock.finance.balance_sheet, "balance_sheet"),
        (stock.finance.cash_flow, "cash_flow"),
    ]:
        # Vòng lặp thử lại (Retry mechanism)
        while True:
            try:
                print(f"   -> Đang tải {name}...", end="")
                df = func(period="quarter")
                
                df["symbol"] = symbol
                df["report"] = name
                all_data.append(df)
                print(" ✅ Xong")
                
                # Ngủ ngắn sau khi thành công
                time.sleep(REQUEST_DELAY)
                break # Thoát vòng lặp while để sang báo cáo tiếp theo
                
            except (Exception, SystemExit) as e:
                # Bắt cả lỗi thường và lỗi SystemExit do rate limit
                print(f"\n⚠️ Gặp lỗi hoặc bị chặn Rate Limit.")
                print("⏳ Đang tạm dừng 65 giây để hồi phục quota...")
                time.sleep(65) # Chờ 65s (hơn 1 phút) để reset limit
                print("▶️ Đang thử lại...")
                # Code sẽ tự quay lại đầu vòng while để gọi lại hàm func()

# Xuất file CSV giữ nguyên số dài
final_df = pd.concat(all_data, ignore_index=True)
final_df.to_csv("financial_report_30_companies_quarter.csv", index=False, encoding='utf-8-sig', float_format='%.0f')

print("\n🎉 Hoàn thành tất cả!")

🔄 Đang xử lý mã: HPG...
   -> Đang tải income_statement... ✅ Xong
   -> Đang tải balance_sheet... ✅ Xong
   -> Đang tải cash_flow... ✅ Xong
🔄 Đang xử lý mã: VCB...
   -> Đang tải income_statement... ✅ Xong
   -> Đang tải balance_sheet... ✅ Xong
   -> Đang tải cash_flow... ✅ Xong
🔄 Đang xử lý mã: SSI...
   -> Đang tải income_statement... ✅ Xong
   -> Đang tải balance_sheet... ✅ Xong
   -> Đang tải cash_flow... ✅ Xong
🔄 Đang xử lý mã: VNM...
   -> Đang tải income_statement... ✅ Xong
   -> Đang tải balance_sheet...
⚠️ 
⚠️  GIỚI HẠN API ĐÃ ĐẠT TỐI ĐA (Rate Limit Exceeded)

📌 Bạn đã đạt giới hạn tối đa số lượt yêu cầu API trong 1 phút (minute).
   (You have reached the maximum API request limit for this period)

📊 Chi tiết (Details):
   • Gói hiện tại: Khách (Guest)
   • Giới hạn: 20 requests/phút
   • Đã sử dụng: 20/20
   • Chờ 35 giây để tiếp tục (Wait to retry)

💡 Giải pháp (Solutions):
   1️⃣ Chờ 35 giây rồi thử lại
      (Wait and retry)
   2️⃣ Tham gia gói thành viên tài trợ để sử dụn

In [5]:
final_df.columns

Index(['ticker', 'yearReport', 'lengthReport', 'Revenue YoY (%)',
       'Revenue (Bn. VND)', 'Attribute to parent company (Bn. VND)',
       'Attribute to parent company YoY (%)', 'Financial Income',
       'Interest Expenses', 'Sales',
       ...
       'Difference upon Assets Revaluation', 'Other Reserves',
       'Profits from other activities',
       'Net Cash Flows from Operating Activities before BIT',
       'Payment from reserves', 'Convertible bonds (Bn. VND)',
       '_Increase/Decrease in receivables', '_Increase/Decrease in payables',
       'Profit/Loss from disposal of fixed assets', 'Leased assets'],
      dtype='str', length=161)

In [6]:
final_df.head()

,ticker,yearReport,lengthReport,Revenue YoY (%),Revenue (Bn. VND),Attribute to parent company (Bn. VND),Attribute to parent company YoY (%),Financial Income,Interest Expenses,Sales,...,Difference upon Assets Revaluation,Other Reserves,Profits from other activities,Net Cash Flows from Operating Activities before BIT,Payment from reserves,Convertible bonds (Bn. VND),_Increase/Decrease in receivables,_Increase/Decrease in payables,Profit/Loss from disposal of fixed assets,Leased assets
0,HPG,2025,4,0.342568,4.730162e+13,3.860994e+12,0.374682,4.373360e+11,-1.236525e+12,4.730162e+13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,HPG,2025,3,0.072697,3.679387e+13,3.988318e+12,0.319348,7.119034e+11,-8.121941e+11,3.679387e+13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,HPG,2025,2,-0.091398,3.628619e+13,4.256487e+12,0.282359,4.981837e+11,-4.391126e+11,3.628619e+13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,HPG,2025,1,0.220569,3.795064e+13,3.344285e+12,0.165017,4.380572e+11,-6.270244e+11,3.795064e+13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,HPG,2024,4,0.008808,3.523220e+13,2.808645e+12,-0.055212,7.005601e+11,-5.624927e+11,3.523220e+13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# xem doanh thu thuần của HPG trong các quý năm 2025
final_df[final_df['symbol'] == 'HPG']





,ticker,yearReport,lengthReport,Revenue YoY (%),Revenue (Bn. VND),Attribute to parent company (Bn. VND),Attribute to parent company YoY (%),Financial Income,Interest Expenses,Sales,...,Difference upon Assets Revaluation,Other Reserves,Profits from other activities,Net Cash Flows from Operating Activities before BIT,Payment from reserves,Convertible bonds (Bn. VND),_Increase/Decrease in receivables,_Increase/Decrease in payables,Profit/Loss from disposal of fixed assets,Leased assets
0,HPG,2025,4,0.342568,4.730162e+13,3.860994e+12,0.374682,4.373360e+11,-1.236525e+12,4.730162e+13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,HPG,2025,3,0.072697,3.679387e+13,3.988318e+12,0.319348,7.119034e+11,-8.121941e+11,3.679387e+13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,HPG,2025,2,-0.091398,3.628619e+13,4.256487e+12,0.282359,4.981837e+11,-4.391126e+11,3.628619e+13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,HPG,2025,1,0.220569,3.795064e+13,3.344285e+12,0.165017,4.380572e+11,-6.270244e+11,3.795064e+13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,HPG,2024,4,0.008808,3.523220e+13,2.808645e+12,-0.055212,7.005601e+11,-5.624927e+11,3.523220e+13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,HPG,2014,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
152,HPG,2013,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
153,HPG,2013,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
154,HPG,2013,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
print(final_df[final_df["ticker"]=="HPG"][["yearReport","lengthReport","Revenue (Bn. VND)"]])

     yearReport  lengthReport  Revenue (Bn. VND)
0          2025             4       4.730162e+13
1          2025             3       3.679387e+13
2          2025             2       3.628619e+13
3          2025             1       3.795064e+13
4          2024             4       3.523220e+13
..          ...           ...                ...
151        2014             1                NaN
152        2013             4                NaN
153        2013             3                NaN
154        2013             2                NaN
155        2013             1                NaN

[156 rows x 3 columns]


In [13]:
df = final_df.dropna(axis=1, how="all")

In [ ]:
df.head()

,ticker,yearReport,lengthReport,Revenue YoY (%),Revenue (Bn. VND),Attribute to parent company (Bn. VND),Attribute to parent company YoY (%),Financial Income,Interest Expenses,Sales,...,Difference upon Assets Revaluation,Other Reserves,Profits from other activities,Net Cash Flows from Operating Activities before BIT,Payment from reserves,Convertible bonds (Bn. VND),_Increase/Decrease in receivables,_Increase/Decrease in payables,Profit/Loss from disposal of fixed assets,Leased assets
0,HPG,2025,4,0.342568,4.730162e+13,3.860994e+12,0.374682,4.373360e+11,-1.236525e+12,4.730162e+13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,HPG,2025,3,0.072697,3.679387e+13,3.988318e+12,0.319348,7.119034e+11,-8.121941e+11,3.679387e+13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,HPG,2025,2,-0.091398,3.628619e+13,4.256487e+12,0.282359,4.981837e+11,-4.391126e+11,3.628619e+13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,HPG,2025,1,0.220569,3.795064e+13,3.344285e+12,0.165017,4.380572e+11,-6.270244e+11,3.795064e+13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,HPG,2024,4,0.008808,3.523220e+13,2.808645e+12,-0.055212,7.005601e+11,-5.624927e+11,3.523220e+13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


: 